In [ ]:
#%pip install kagglehub[pandas-datasets]

# 1. Pré-processamento

##1.1 Pré-processamento Steam Games

In [ ]:
import pandas as pd

# Importamos dados da steam direto do huggingface
df_steam = pd.read_parquet("hf://datasets/FronkonGames/steam-games-dataset/data/train-00000-of-00001.parquet")

###Funções auxiliares:

In [ ]:
import ast
import html
import re
import numpy as np
import pandas as pd

_ITEM = re.compile(r"'((?:[^'\\]|\\.)*)'|\"((?:[^\"\\]|\\.)*)\"")

def parse_list(v):
    """Converte repr de numpy, lista stringificada ou iterável em list[str]."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return []
    if isinstance(v, (list, tuple, np.ndarray)):
        return [str(x).strip() for x in v if str(x).strip()]
    return [(a or b).strip() for a, b in _ITEM.findall(str(v)) if (a or b).strip()]

def parse_tags(v):
    """Converte dict {tag: votos}, lista ou texto em list[str] ordenada por votos."""
    if isinstance(v, dict):
        return [str(k) for k, _ in sorted(v.items(), key=lambda kv: -(kv[1] or 0))]
    if isinstance(v, str) and v.strip().startswith("{"):
        try:
            return parse_tags(ast.literal_eval(v))
        except (ValueError, SyntaxError):
            pass
    return parse_list(v)

def clean_text(v):
    if not isinstance(v, str):
        return np.nan
    v = html.unescape(re.sub(r"<[^>]+>", " ", v))
    return re.sub(r"\s+", " ", v).strip() or np.nan

def _hit(col, rx):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return df[col].fillna("").astype(str).str.contains(rx).astype(bool)

###Fluxo de limpeza dos dados

#### Configurações e Parâmetros

Colunas a serem excluidas, colunas com o mesmo padrão de dados, genêros não desejados (poucos registros ou não acho apropriado serem considerados no conjunto de dados)


In [ ]:
STRICT = True
DEDUP_NAMES = True

COLS_DROP = [
    "score_rank", "user_score", "movies", "screenshots", "packages",
    "support_email", "support_url", "metacritic_url", "__index_level_0__", "Unnamed: 0",
]

# Gêneros removidos só da lista (o registro continua)
GENRES_DROP = {
    "Early Access", "Free To Play", "Massively Multiplayer",
    "360 Video", "Tutorial", "Episodic",
    "Accounting", "Movie", "Documentary", "Short",
}

# Gêneros que indicam que o registro NÃO é um jogo -> remove a LINHA
GENRES_DROP_ROW = {
    "Utilities", "Video Production", "Photo Editing", "Design & Illustration",
    "Animation & Modeling", "Audio Production", "Software Training",
    "Web Publishing", "Game Development", "Education",
}

# Gêneros de conteúdo adulto -> remove a LINHA
GENRES_ADULT = {"Nudity", "Sexual Content"}

# Critérios adicionais de conteúdo impróprio (usados no passo 2.4b)
ADULT_MIN_AGE = None      # required_age >= isso é removido; None para desligar
ADULT_EXPLICIT = re.compile(   # nome, descrições e notes
    r"\b(?:hentai|porn\w*|nsfw|erotic\w*|uncensored|genitalia|lewd|fetish\w*|bdsm|"
    r"explicit(?:ly)?\s+sex\w*|sex\s+scenes?|adults?[- ]only|18\+)", re.I)
ADULT_NOTES = re.compile(      # só em notes (aviso de conteúdo do desenvolvedor)
    r"\b(?:nud(?:e|ity)|sexual\w*|sex\s+scenes?)\b", re.I)

LIST_COLS = [
    "supported_languages", "full_audio_languages",
    "developers", "publishers", "categories", "genres",
]

TEXT_COLS = ["detailed_description", "short_description", "about_the_game", "notes", "reviews"]
NOISE_NAME = re.compile(r"\b(?:playtest|demo|soundtrack|ost|dlc|beta)\b", re.I)

#### Pipeline de tratamento

In [ ]:
df = df_steam.copy()
log = {"linhas_entrada": len(df)}

# 2.1) Chave appID, nomes válidos e deduplicação pelo registro mais completo
df["appID"] = pd.to_numeric(df["appID"], errors="coerce")
df["name"] = df["name"].astype("string").str.strip()
df = df[df["appID"].notna() & df["name"].notna() & (df["name"] != "")].copy()
df["appID"] = df["appID"].astype("int64")

rico = df.notna().sum(axis=1) + df.select_dtypes("number").ne(0).sum(axis=1)
df = (
    df.assign(_rico=rico)
    .sort_values("_rico", ascending=False, kind="stable")
    .drop_duplicates(subset="appID", keep="first")
    .drop(columns="_rico")
    .sort_index()
)
log["apos_appID_e_nomes_validos"] = len(df)

# 2.2) Remoção de colunas inúteis e parsing de datas / donos
df = df.drop(columns=COLS_DROP, errors="ignore")

df["release_date"] = pd.to_datetime(df["release_date"], format="mixed", errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

own = df["estimated_owners"].astype(str).str.extract(r"(\d+)\s*-\s*(\d+)").astype(float)
df["owners_min"], df["owners_max"] = own[0], own[1]
df["owners_mid"] = (df["owners_min"] + df["owners_max"]) / 2
df = df.drop(columns=["estimated_owners"])

# 2.3) Listas e filtragem de gêneros indesejados
for c in set(LIST_COLS).intersection(df.columns):
    df[c] = df[c].map(parse_list)

# Remoção de LINHAS (não são jogos / gênero adulto), antes de limpar as tags
df = df[~df["genres"].map(lambda gs: bool(GENRES_DROP_ROW.intersection(gs)))].copy()
log["apos_remocao_nao_jogos"] = len(df)

df = df[~df["genres"].map(lambda gs: bool(GENRES_ADULT.intersection(gs)))].copy()
log["apos_remocao_generos_adultos"] = len(df)

# Remoção de TAGS indesejadas (o registro continua)
df["genres"] = df["genres"].map(lambda gs: [g for g in gs if g not in GENRES_DROP])
df["n_genres"] = df["genres"].map(len)
df["n_languages"] = df["supported_languages"].map(len)

df = df[df["n_genres"] > 0].copy()
log["apos_filtro_generos"] = len(df)

# 2.4) Limpeza de texto, tags e features derivadas
for c in set(TEXT_COLS).intersection(df.columns):
    df[c] = df[c].map(clean_text)

if "tags" in df.columns:
    df["tags"] = df["tags"].map(parse_tags)
    if df["tags"].map(len).sum() == 0:
        df = df.drop(columns="tags")

df["metacritic_score"] = df["metacritic_score"].replace(0, np.nan)
df["is_free"] = df["price"].fillna(0).eq(0)
df["total_reviews"] = df["positive"].fillna(0) + df["negative"].fillna(0)
df["has_reviews"] = df["total_reviews"] > 0
df["positive_ratio"] = (df["positive"] / df["total_reviews"]).where(df["has_reviews"])
df["is_demo_playtest"] = df["name"].str.contains(NOISE_NAME, na=False)
df["n_platforms"] = df[["windows", "mac", "linux"]].sum(axis=1)

# 2.4b) Conteúdo impróprio (sexual/obsceno): texto, notes e faixa etária
explicito = pd.Series(False, index=df.index)
for c in ["name", "short_description", "detailed_description", "notes"]:
    explicito |= _hit(c, ADULT_EXPLICIT)

aviso_notes = _hit("notes", ADULT_NOTES)
por_idade = (pd.Series(False, index=df.index) if ADULT_MIN_AGE is None
             else df["required_age"].fillna(0).ge(ADULT_MIN_AGE))

log["removidos_por_texto_explicito"] = int(explicito.sum())
log["removidos_por_aviso_notes"] = int(aviso_notes.sum())
log["removidos_por_idade"] = int(por_idade.sum())

df = df[~(explicito | aviso_notes | por_idade)].copy()
log["apos_filtro_conteudo_improprio"] = len(df)

# 2.5) Filtro strict (antes de deduplicar nomes para evitar trabalho extra)
if STRICT:
    df = df[df["short_description"].notna() & ~df["is_demo_playtest"]].copy()
    log["apos_filtro_strict"] = len(df)

# 2.6) Possíveis duplicados com appID diferente (mesmo nome normalizado + mesmos devs)
nome_norm = (
    df["name"].str.lower()
    .str.replace(r"[®™©]", "", regex=True)
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)
dev_key = df["developers"].map(lambda xs: "|".join(sorted(x.lower() for x in xs)))
chave = nome_norm + "||" + dev_key
elegivel = dev_key.ne("") & nome_norm.ne("")
df["is_possible_dup"] = chave.duplicated(keep=False) & elegivel

if DEDUP_NAMES:
    ordem = df.assign(_k=chave).sort_values(["total_reviews", "owners_max"], ascending=False, kind="stable")
    manter = ~(ordem["_k"].duplicated(keep="first") & elegivel.loc[ordem.index])
    df = df.loc[ordem.index[manter]].sort_index().copy()
    df["is_possible_dup"] = False
    log["apos_dedup_nomes"] = len(df)

# 2.7) Texto único para embeddings / TF-IDF (vetorizado via zip, muito mais rápido que .apply)
tags_col = df["tags"] if "tags" in df.columns else [[] for _ in range(len(df))]
desc_col = df["short_description"].fillna("")

df["texto_modelo"] = [
    ". ".join(p for p in [nome, ", ".join(gens), ", ".join(tgs[:10]), desc] if p)
    for nome, gens, tgs, desc in zip(df["name"], df["genres"], tags_col, desc_col)
]

df_tratado = df.reset_index(drop=True)
log["linhas_saida"], log["colunas_saida"] = df_tratado.shape

for k, v in log.items():
    print(f"{k}: {v}")

linhas_entrada: 124146
apos_appID_e_nomes_validos: 124145
apos_remocao_nao_jogos: 122269
apos_remocao_generos_adultos: 122131
apos_filtro_generos: 113458
removidos_por_texto_explicito: 5494
removidos_por_aviso_notes: 9490
removidos_por_idade: 0
apos_filtro_conteudo_improprio: 103158
apos_filtro_strict: 103012
apos_dedup_nomes: 102868
linhas_saida: 102868
colunas_saida: 45


#### Salvar arquivos .parquet

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

PASTA = "/content/drive/MyDrive/projeto_ia/data"
os.makedirs(PASTA, exist_ok=True)

df_tratado.to_parquet(f"{PASTA}/steam_tratado.parquet", index=False)
print("salvo:", df_tratado.shape)

Mounted at /content/drive
salvo: (102868, 45)


In [ ]:
# Exemplo quando a coluna vem como string "[ação, comédia]"
contagem = (
    df_tratado["genres"]
    .dropna()
    #.str.strip("[]")
    #.str.split(",")
    .explode()
    .str.strip()
    #.loc[lambda s: s != ""]  # remove strings vazias caso existam
    .value_counts(normalize=False)
)

print(contagem)

genres
Indie         74523
Casual        44952
Action        44186
Adventure     41020
Strategy      21608
Simulation    21349
RPG           18382
Sports         4697
Racing         4036
Violent         343
Gore            215
Name: count, dtype: int64


##1.2 Pré-processamento dataset Filmes

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "final_dataset.csv"

# Load the latest version
df_movies = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "raedaddala/top-500-600-movies-of-each-year-from-1960-to-2024",
  file_path
)

100%|██████████| 43.8M/43.8M [00:01<00:00, 24.5MB/s]


###Funções auxiliares

In [ ]:
import re
import numpy as np
import pandas as pd

def parse_votes(v):
    """'2.5K' -> 2500, '1.2M' -> 1200000, '114' -> 114."""
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)
    m = re.fullmatch(r"(\d+(?:\.\d+)?)\s*([KMB]?)", str(v).strip().upper().replace(",", ""))
    if not m:
        return np.nan
    num, suf = m.groups()
    return round(float(num) * {"": 1, "K": 1e3, "M": 1e6, "B": 1e9}[suf])

def parse_duration(v):
    """'1h 16m' / '1h16min' -> 76, '2h' -> 120, '45m' -> 45."""
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).lower()
    h = re.search(r"(\d+)\s*h", s)
    m = re.search(r"(\d+)\s*m", s)
    if not (h or m):
        return np.nan
    return (int(h.group(1)) if h else 0) * 60 + (int(m.group(1)) if m else 0)

def aplicar(df, col, func):
    """Converte a coluna e avisa se algum valor não nulo falhou na conversão."""
    novo = df[col].map(func)
    falhou = df[col].notna() & novo.isna()
    if falhou.any():
        print(f"{col}: {falhou.sum()} valores não convertidos, ex.: {df.loc[falhou, col].unique()[:5]}")
    df[col] = novo

###Limpeza e tratamento dos dados

In [ ]:
COLS_DROP = ["opening_weekend_gross", "gross_us_canada", "filming_locations", "gross_worldwide", "budget", "awards_content"]
LIST_COLS = ["writers", "genres", "directors", "stars", "production_companies", "languages", "countries_origin"]
STRG_COLS = ["id", "title", "description", "movie_link", "mpa"]
DATE_COLS = ["release_date"]
NUMS_COLS = ["duration", "rating", "votes", "méta_score"]

df_movies.drop(columns=COLS_DROP, inplace=True, errors='ignore')

# Exclusão de ids (PK) vazios e ids duplicados
df_movies['id'].dropna()
df_movies['id'].drop_duplicates(inplace=True)

aplicar(df_movies, "votes", parse_votes)
aplicar(df_movies, "duration", parse_duration)

# dtypes conforme as listas
for c in STRG_COLS:
    df_movies[c] = df_movies[c].astype("string")

for c in DATE_COLS:
    df_movies[c] = pd.to_datetime(df_movies[c], errors="coerce")

for c in LIST_COLS:
    df_movies[c] = df_movies[c].map(parse_list)      # list[str]; no pandas fica como object

df_movies["rating"] = pd.to_numeric(df_movies["rating"], errors="coerce").astype("float64")
for c in ["duration", "votes", "méta_score"]:
    df_movies[c] = pd.to_numeric(df_movies[c], errors="coerce").round().astype("Int64")

print(df_movies[STRG_COLS + DATE_COLS + LIST_COLS + NUMS_COLS].dtypes)

duration: 88 valores não convertidos, ex.: ['R' 'Passed' 'PG-13' 'PG' '18+']
id                      string[python]
title                   string[python]
description             string[python]
movie_link              string[python]
mpa                     string[python]
release_date            datetime64[ns]
writers                         object
genres                          object
directors                       object
stars                           object
production_companies            object
languages                       object
countries_origin                object
duration                         Int64
rating                         float64
votes                            Int64
méta_score                       Int64
dtype: object


###Salvar arquivos .parquet

In [ ]:
import os

PASTA = "/content/drive/MyDrive/projeto_ia/data"
os.makedirs(PASTA, exist_ok=True)

df_movies.to_parquet(f"{PASTA}/filmes_tratado.parquet", index=False)
print("salvo:", df_movies.shape)

salvo: (63249, 17)


#2. Desenvolvimento do modelo

In [ ]:
import pandas as pd

df_steam = pd.read_parquet("/content/drive/MyDrive/projeto_ia/data/steam_tratado.parquet")
if df_steam is not None:
  print(f'LOG: Dados steam tratados para treinamento carregados')
  display(df_steam.head())

LOG: Dados steam tratados para treinamento carregados


,appID,name,release_date,peak_ccu,required_age,price,dlc_count,detailed_description,short_description,supported_languages,...,n_genres,n_languages,is_free,total_reviews,has_reviews,positive_ratio,is_demo_playtest,n_platforms,is_possible_dup,texto_modelo
0,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,0,0,5.24,0,"Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",[English],...,1,1,False,255,True,0.988235,False,1,False,Supipara - Chapter 1 Spring Has Come!. Adventu...
1,1034400,Mystery Solitaire The Black Raven,2019-05-06,0,0,4.99,0,"Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,"[English, French, German, Russian]",...,1,4,False,24,True,0.875000,False,2,False,Mystery Solitaire The Black Raven. Casual. Dis...
2,3631080,Maze Quest VR,2025-04-24,0,0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,[English],...,1,1,False,0,False,NaN,False,1,False,Maze Quest VR. Action. Its not just a Maze; it...
3,1934300,Armored Brigade II,2025-04-08,8,0,35.99,1,Building on the success of its acclaimed prede...,Armored Brigade II revolutionizes real-time ta...,[English],...,2,1,False,130,True,0.900000,False,1,False,"Armored Brigade II. Simulation, Strategy. Armo..."
4,700340,Galacatraz: Eject Equip Escape,2018-02-01,0,0,0.99,0,"Years ago, Ricardo Luger was in the wrong plac...","Escape Galacatraz, the universe's most secure ...",[English],...,2,1,False,0,False,NaN,False,1,False,"Galacatraz: Eject Equip Escape. Action, Indie...."


## 2.1 Configuração

Bibliotecas usadas e parâmetros ajustáveis do modelo, centralizados aqui para facilitar experimentos.

| Parâmetro | Função |
|---|---|
| `PESO_GENEROS` | Quantas vezes gêneros/tags se repetem no texto, para terem mais peso que a descrição |
| `MIN_DOC_POR_BASE` | Um termo só entra no vocabulário se aparecer em pelo menos N jogos **e** N filmes |
| `MIN_FILMES_POR_GENERO` | Gêneros com poucos exemplos ficam fora do classificador |
| `MIN_VOTOS_FILME` | Filmes com menos votos não são recomendados (evita filmes obscuros) |
| `ALPHA` | Peso da similaridade de texto vs. similaridade de gêneros no score final |
| `BETA` | Peso do bônus de qualidade (nota IMDb) no score final |

In [ ]:
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

CAMINHO_JOGOS  = "/content/drive/MyDrive/projeto_ia/data/steam_tratado.parquet"
CAMINHO_FILMES = "/content/drive/MyDrive/projeto_ia/data/filmes_tratado.parquet"

PESO_GENEROS          = 2     # quantas vezes gêneros/tags se repetem no texto
MIN_DOC_POR_BASE      = 5     # termo precisa aparecer em >= 5 jogos E >= 5 filmes
MIN_FILMES_POR_GENERO = 200   # gêneros raros ficam fora do classificador
MIN_VOTOS_FILME       = 1000  # filmes com menos votos não são recomendados
ALPHA                 = 0.5   # peso do texto vs. gêneros (1 = só texto, 0 = só gêneros)
BETA                  = 0.1   # peso do bônus de qualidade do filme
MIN_TERMOS            = 3     # linhas com menos termos que isso são removidas (0 = vazia)

In [ ]:
_ITEM = re.compile(r"'([^']*)'|\"([^\"]*)\"")

def como_lista(v):
    # Converte array, lista ou string "['a', 'b']" em list[str].
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return []
    if isinstance(v, str):
        itens = [(a or b).strip() for a, b in _ITEM.findall(v)]
        return [x for x in itens if x] or ([v.strip()] if v.strip() else [])
    return [str(x).strip() for x in v if str(x).strip()]

jogos  = pd.read_parquet(CAMINHO_JOGOS).reset_index(drop=True)
filmes = pd.read_parquet(CAMINHO_FILMES)

# Proteção: garante 1 linha por filme e só filmes com descrição
filmes = (filmes.dropna(subset=["id"])
                .drop_duplicates(subset="id")
                .loc[lambda d: d["description"].notna()]
                .reset_index(drop=True))

for col in ["genres", "tags"]:
    if col in jogos.columns:
        jogos[col] = jogos[col].map(como_lista)
filmes["genres"] = filmes["genres"].map(como_lista)

print(f"Jogos: {len(jogos):,} | Filmes: {len(filmes):,}")

Jogos: 102,868 | Filmes: 60,889


## 2.3 Construção do texto de cada item

Cada jogo e cada filme é representado por um único texto, que será vetorizado.

- **Jogo:** gêneros + 10 tags mais votadas + descrição curta.
- **Filme:** gêneros principais + descrição (sinopse).

Decisões de projeto:
- **O nome não entra no texto**, pois não descreve o conteúdo e gera falsos pareamentos (ex.: *Dark Souls* × *Dark Shadows*).
- **Gêneros/tags são repetidos** (`PESO_GENEROS`) para não serem "afogados" pela descrição.
- **As tags da Steam são a principal ponte com os filmes**: rótulos como *Horror*, *Sci-fi* e *Western* existem nos dois domínios, ao contrário dos gêneros da Steam (*Indie*, *Casual*, *RPG*).

In [ ]:
def montar_texto(generos, tags, descricao, peso=PESO_GENEROS):
    rotulos = ", ".join(list(generos) + list(tags))
    partes = [rotulos] * peso + [descricao or ""]
    return ". ".join(p for p in partes if p)

tags_jogos = jogos["tags"] if "tags" in jogos.columns else [[]] * len(jogos)
jogos["texto"] = [
    montar_texto(g, t[:10], d)
    for g, t, d in zip(jogos["genres"], tags_jogos, jogos["short_description"])
]
filmes["texto"] = [
    montar_texto(g, [], d)
    for g, d in zip(filmes["genres"], filmes["description"])
]

print("Exemplo jogo :", jogos["texto"].iloc[0][:200])
print("Exemplo filme:", filmes["texto"].iloc[0][:200])

Exemplo jogo : Adventure. Adventure. Spring has come, and our protagonist, Yukinari, returns home after 7 years. Between his second cousin Sakura, who meets him outside in a full maid outfit, Hotaru, a half-Japanese
Exemplo filme: Drama. Drama. An extortion ring murders anyone who refuses to pay protection money. New Scotland Yard is on the case, but a victim is killed while under their protection. A sexy secretary falls under 


## 2.4 Vetorização com TF-IDF

Transforma cada texto em um vetor numérico. Cada posição do vetor corresponde a um termo, com peso:

- **TF (term frequency):** frequência do termo no texto (com escala logarítmica, `sublinear_tf`);
- **IDF (inverse document frequency):** raridade do termo no conjunto. Termos muito comuns valem pouco; termos específicos valem mais.

O vetorizador é ajustado (`fit`) com **jogos e filmes juntos**, para que os dois compartilhem o mesmo vocabulário. Só assim os vetores de jogos e filmes são comparáveis.

São usados unigramas e bigramas (`ngram_range=(1, 2)`), para capturar expressões como *"serial killer"* e *"space station"*.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",       # ignora "the", "and", "of"...
    ngram_range=(1, 2),         # palavras soltas e pares ("serial killer")
    min_df=3,                   # ignora termos em menos de 3 textos (erros, nomes próprios)
    max_df=0.5,                 # ignora termos em mais de 50% dos textos (genéricos)
    max_features=200_000,       # limite de tamanho do vocabulário
    sublinear_tf=True,          # usa log da contagem: repetir 10x não vale 10x mais
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",   # só palavras com letras
)

corpus = pd.concat([jogos["texto"], filmes["texto"]], ignore_index=True)
vectorizer.fit(corpus)

X_jogos_full  = vectorizer.transform(jogos["texto"])
X_filmes_full = vectorizer.transform(filmes["texto"])

print("Matriz jogos :", X_jogos_full.shape)
print("Matriz filmes:", X_filmes_full.shape)

Matriz jogos : (102868, 189867)
Matriz filmes: (60889, 189867)


In [ ]:
df_jogos  = np.asarray((X_jogos_full  > 0).sum(axis=0)).ravel()   # em quantos jogos cada termo aparece
df_filmes = np.asarray((X_filmes_full > 0).sum(axis=0)).ravel()   # em quantos filmes cada termo aparece
compartilhado = (df_jogos >= MIN_DOC_POR_BASE) & (df_filmes >= MIN_DOC_POR_BASE)

X_jogos  = normalize(X_jogos_full[:, compartilhado])
X_filmes = normalize(X_filmes_full[:, compartilhado])
vocab    = vectorizer.get_feature_names_out()[compartilhado]

termos = vectorizer.get_feature_names_out()
so_jogos = termos[(df_jogos >= 50) & (df_filmes == 0)]
print(f"Vocabulário total: {len(termos):,} -> compartilhado: {len(vocab):,}")
print("Exemplos de termos só de jogos (removidos):", list(so_jogos[:15]))

Vocabulário total: 189,867 -> compartilhado: 13,916
Exemplos de termos só de jogos (removidos): ['achievements quick', 'action arcade', 'action casual', 'action experience', 'action indie', 'action platformer', 'action puzzle', 'action racing', 'action rogue', 'action roguelike', 'action roguelite', 'action rpg', 'action shooter', 'action shooting', 'action simulation']


In [ ]:
def termos_por_linha(X):
    """Quantos termos não nulos cada linha da matriz tem."""
    return np.asarray(X.getnnz(axis=1)).ravel()

def relatorio(nome, nnz):
    print(f"{nome}: {len(nnz):,} linhas | vazias: {(nnz == 0).sum():,} | "
          f"fracas (< {MIN_TERMOS} termos): {(nnz < MIN_TERMOS).sum():,} | "
          f"mediana: {np.median(nnz):.0f}")

# 1) Estado antes da limpeza
nnz_j = termos_por_linha(X_jogos)
nnz_f = termos_por_linha(X_filmes)
relatorio("Jogos  (antes) ", nnz_j)
relatorio("Filmes (antes) ", nnz_f)

# 2) Máscaras: True = a linha fica
manter_j = nnz_j >= MIN_TERMOS
manter_f = nnz_f >= MIN_TERMOS

# 3) Amostra do que vai ser removido (para conferir antes de aplicar)
cols_j = [c for c in ("name", "texto") if c in jogos.columns]
cols_f = [c for c in ("title", "description") if c in filmes.columns]
print("\nExemplos de jogos removidos:");  print(jogos.loc[~manter_j, cols_j].head(8).to_string())
print("\nExemplos de filmes removidos:"); print(filmes.loc[~manter_f, cols_f].head(8).to_string())

# 4) Aplica a MESMA máscara no DataFrame e nas duas matrizes de cada base
jogos         = jogos.loc[manter_j].reset_index(drop=True)
X_jogos       = X_jogos[manter_j]
X_jogos_full  = X_jogos_full[manter_j]

filmes        = filmes.loc[manter_f].reset_index(drop=True)
X_filmes      = X_filmes[manter_f]
X_filmes_full = X_filmes_full[manter_f]

# 5) Confere o alinhamento e mostra o resultado
assert len(jogos)  == X_jogos.shape[0]  == X_jogos_full.shape[0]
assert len(filmes) == X_filmes.shape[0] == X_filmes_full.shape[0]
print(f"\nRemovidos: {(~manter_j).sum():,} jogos | {(~manter_f).sum():,} filmes")
relatorio("Jogos  (depois)", termos_por_linha(X_jogos))
relatorio("Filmes (depois)", termos_por_linha(X_filmes))

Jogos  (antes) : 102,868 linhas | vazias: 49 | fracas (< 3 termos): 1,099 | mediana: 19
Filmes (antes) : 60,889 linhas | vazias: 0 | fracas (< 3 termos): 14 | mediana: 18

Exemplos de jogos removidos:
                                                  name                                                                                                                                                                                                                                                                                                       texto
38                                             謎解き二十部屋                                                      Casual, Indie, RPG. Casual, Indie, RPG. RPGツクールMZ製の謎解きゲームです。一部屋完結型（例外あり）の謎解きを20部屋分用意しました。 ホラー要素は一切無し、ただしストーリーもシナリオもありません、じっくり、のんびりと謎解きをお楽しみ下さい。 簡単な部屋はひらめきでクリア出来るでしょうが、難しい部屋はメモ等が必要になるかも知れません。 順調に進めば1～２時間位でクリア出来る…ハズ… ツクール製ならではの謎も用意しましたので、是非お楽しみ下さい。
45                                               筠宝大冒险                                   

In [ ]:
# Rótulos: só gêneros com exemplos suficientes
generos_filmes = filmes["genres"]
contagem = pd.Series([g for gs in generos_filmes for g in gs]).value_counts()

GENEROS_PRINCIPAIS = {
    "Action", "Adventure", "Animation", "Biography", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History", "Horror",
    "Music", "Musical", "Mystery", "Romance", "Sci-Fi", "Sport",
    "Thriller", "War", "Western",
}

generos_validos = set(contagem[contagem >= MIN_FILMES_POR_GENERO].index) & GENEROS_PRINCIPAIS

mlb = MultiLabelBinarizer(classes=sorted(generos_validos))
Y = mlb.fit_transform(generos_filmes.map(lambda gs: [g for g in gs if g in generos_validos]))
tem_genero = Y.sum(axis=1) > 0
print(f"{len(mlb.classes_)} gêneros no classificador:", list(mlb.classes_))

# Entrada: SÓ a descrição (sem os gêneros, para não vazar a resposta)
X_desc = normalize(vectorizer.transform(filmes["description"].fillna(""))[:, compartilhado])

# NOVO: descarta filmes cuja descrição ficou vazia/fraca no vocabulário compartilhado
nnz_d = termos_por_linha(X_desc)
tem_genero = tem_genero & (nnz_d >= MIN_TERMOS)
print(f"Filmes usados no classificador: {tem_genero.sum():,} de {len(filmes):,} "
      f"(descrição fraca/vazia: {(nnz_d < MIN_TERMOS).sum():,})")

X_tr, X_te, Y_tr, Y_te = train_test_split(
    X_desc[tem_genero], Y[tem_genero], test_size=0.2, random_state=42
)

def novo_classificador():
    return OneVsRestClassifier(
        LogisticRegression(C=4.0, class_weight="balanced", solver="liblinear", max_iter=1000),
        n_jobs=-1,
    )

clf = novo_classificador().fit(X_tr, Y_tr)
Y_pred = clf.predict(X_te)

print(f"F1 micro: {f1_score(Y_te, Y_pred, average='micro'):.3f}")
print(f"F1 macro: {f1_score(Y_te, Y_pred, average='macro'):.3f}")
print(classification_report(Y_te, Y_pred, target_names=mlb.classes_, zero_division=0))

21 gêneros no classificador: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sport', 'Thriller', 'War', 'Western']
Filmes usados no classificador: 60,448 de 60,875 (descrição fraca/vazia: 97)
F1 micro: 0.534
F1 macro: 0.483
              precision    recall  f1-score   support

      Action       0.39      0.66      0.49      1659
   Adventure       0.34      0.63      0.44      1318
   Animation       0.24      0.44      0.31       254
   Biography       0.26      0.57      0.35       456
      Comedy       0.54      0.67      0.60      3679
       Crime       0.46      0.69      0.55      1875
 Documentary       0.53      0.77      0.63       574
       Drama       0.73      0.69      0.71      6944
      Family       0.23      0.51      0.32       580
     Fantasy       0.31      0.58      0.41       681
     History       0.22      0.51      0.

In [ ]:
# Entrada dos jogos para o classificador: SÓ a short_description
# (não usar X_jogos: o "texto" dela contém gêneros/tags do Steam e tornaria a validação circular)
X_desc_jogos = normalize(
    vectorizer.transform(jogos["short_description"].fillna(""))[:, compartilhado]
)

# Linhas com pouca informação: só marcamos, NÃO apagamos do DataFrame
nnz_dj = termos_por_linha(X_desc_jogos)
usar_jogo = nnz_dj >= MIN_TERMOS          # máscara usada apenas na avaliação

assert X_desc_jogos.shape[0] == len(jogos)   # alinhamento com o DataFrame

print(f"Jogos com descrição utilizável: {usar_jogo.sum():,} de {len(jogos):,} ({usar_jogo.mean():.1%})")
print(f"vazias: {(nnz_dj == 0).sum():,} | fracas (< {MIN_TERMOS} termos): {(nnz_dj < MIN_TERMOS).sum():,}")
print(f"termos por linha -> mediana: {np.median(nnz_dj):.0f} | "
      f"p5: {np.percentile(nnz_dj, 5):.0f} | p95: {np.percentile(nnz_dj, 95):.0f}")

# Amostra do que ficará de fora da avaliação
cols = [c for c in ("name", "short_description") if c in jogos.columns]
print("\nExemplos de jogos fora da avaliação:")
print(jogos.loc[~usar_jogo, cols].head(8).to_string())

Jogos com descrição utilizável: 100,205 de 101,769 (98.5%)
vazias: 688 | fracas (< 3 termos): 1,564
termos por linha -> mediana: 17 | p5: 5 | p95: 26

Exemplos de jogos fora da avaliação:
                                                                  name                                                                                                                                                                                                                    short_description
34                                                             Hapless                                                                                                                                                                                                       This is a turret defense game.
122                                                      Synth Swinger                                                                                                                                              

In [ ]:
scores = clf.decision_function(X_desc_jogos[usar_jogo])

scores.shape

(100205, 21)

In [ ]:
# ---------------- PARÂMETROS DESTA CÉLULA ----------------
TOP_TAGS    = 10     # usa só as N primeiras tags de cada jogo (None = todas)
MIN_POS_VAL = 30     # abaixo disso o gênero é marcado como "poucos positivos"

# gênero do IMDb -> nomes equivalentes no Steam (comparação ignora maiúsculas/minúsculas)
MAPA_ROTULOS = {
    # gêneros do Steam (jogos["genres"]): todo jogo tem esse campo
    "Action":    {"genres": ["Action"]},
    "Adventure": {"genres": ["Adventure"]},
    "Sport":     {"genres": ["Sports"]},
    # tags do Steam (jogos["tags"]): só valem para jogos que TÊM tags
    "Horror":    {"tags": ["Horror"]},
    "Sci-Fi":    {"tags": ["Sci-fi"]},
    "Fantasy":   {"tags": ["Fantasy"]},
    "War":       {"tags": ["War", "World War II", "World War I"]},
    "Western":   {"tags": ["Western"]},
    "Mystery":   {"tags": ["Mystery", "Detective"]},
    "Crime":     {"tags": ["Crime"]},
    "Comedy":    {"tags": ["Comedy", "Funny"]},
    "Romance":   {"tags": ["Romance"]},
    "History":   {"tags": ["Historical"]},
    "Family":    {"tags": ["Family Friendly"]},
}

# ---------------- 1) INSPEÇÃO: confira se os nomes do mapa existem nos dados ----------------
tem_col_tags = "tags" in jogos.columns
freq_gen = pd.Series([g for gs in jogos["genres"] for g in gs]).value_counts()
print("Gêneros do Steam:\n", freq_gen.head(25).to_string())

if tem_col_tags:
    tags_ok = jogos["tags"].map(len).gt(0).to_numpy()          # jogo tem ao menos 1 tag?
    freq_tags = pd.Series([t for ts in jogos["tags"] for t in ts]).value_counts()
    print(f"\nTags: {tags_ok.mean():.1%} dos jogos têm ao menos 1 tag ({tags_ok.sum():,})")
    print("Tags mais frequentes:\n", freq_tags.head(40).to_string())
else:
    tags_ok = np.zeros(len(jogos), dtype=bool)
    freq_tags = pd.Series(dtype=int)
    print("\nColuna 'tags' não existe: só os rótulos baseados em gêneros serão montados.")

freq_gen_l  = {k.lower(): v for k, v in freq_gen.items()}
freq_tags_l = {k.lower(): v for k, v in freq_tags.items()}

# ---------------- 2) MONTAGEM DOS RÓTULOS ----------------
def tem_algum(lista, alvos):
    return any(str(x).lower() in alvos for x in lista)

colunas, rotulavel, fonte, nomes_usados = {}, {}, {}, {}
classes = list(mlb.classes_)

for g_imdb, regra in MAPA_ROTULOS.items():
    if g_imdb not in classes:
        print(f"[ignorado] {g_imdb}: não está em mlb.classes_")
        continue

    if "genres" in regra:
        alvos = {x.lower() for x in regra["genres"]}
        colunas[g_imdb]   = np.array([tem_algum(gs, alvos) for gs in jogos["genres"]])
        rotulavel[g_imdb] = np.ones(len(jogos), dtype=bool)
        fonte[g_imdb], freq_ref = "genres", freq_gen_l
    elif "tags" in regra:
        if not tem_col_tags:
            print(f"[ignorado] {g_imdb}: sem coluna de tags")
            continue
        alvos = {x.lower() for x in regra["tags"]}
        base = jogos["tags"].map(lambda ts: ts[:TOP_TAGS] if TOP_TAGS else ts)
        colunas[g_imdb]   = np.array([tem_algum(ts, alvos) for ts in base])
        rotulavel[g_imdb] = tags_ok          # sem tags -> NÃO dá para dizer que é negativo
        fonte[g_imdb], freq_ref = "tags", freq_tags_l
    else:
        continue

    nomes_usados[g_imdb] = ", ".join(sorted(alvos))
    faltando = [a for a in alvos if freq_ref.get(a, 0) == 0]
    if faltando:
        print(f"[atenção] {g_imdb}: nomes NÃO encontrados nos dados -> {faltando}")

Y_val     = pd.DataFrame(colunas,   index=jogos.index).astype(int)   # 0/1 por jogo e gênero
rotulavel = pd.DataFrame(rotulavel, index=jogos.index)               # True = rótulo confiável
assert len(Y_val) == len(jogos) == len(rotulavel)

# ---------------- 3) RESUMO (considera só jogos com descrição utilizável) ----------------
avaliavel = rotulavel.loc[:, Y_val.columns].to_numpy() & usar_jogo[:, None]
linhas = []
for j, g in enumerate(Y_val.columns):
    n_aval = int(avaliavel[:, j].sum())
    n_pos  = int(Y_val[g].to_numpy()[avaliavel[:, j]].sum())
    linhas.append({
        "genero_imdb": g, "fonte": fonte[g], "nomes_steam": nomes_usados[g],
        "idx_score": classes.index(g), "n_avaliaveis": n_aval, "positivos": n_pos,
        "prevalencia": n_pos / n_aval if n_aval else np.nan,
        "ok": "ok" if n_pos >= MIN_POS_VAL else "poucos positivos",
    })
resumo_rotulos = pd.DataFrame(linhas).set_index("genero_imdb")
print("\n", resumo_rotulos.to_string(float_format=lambda v: f"{v:.3f}"))

Gêneros do Steam:
 Indie         73963
Casual        44663
Action        44065
Adventure     40673
Strategy      21433
Simulation    21227
RPG           17889
Sports         4692
Racing         4030
Violent         343
Gore            215

Coluna 'tags' não existe: só os rótulos baseados em gêneros serão montados.
[ignorado] Horror: sem coluna de tags
[ignorado] Sci-Fi: sem coluna de tags
[ignorado] Fantasy: sem coluna de tags
[ignorado] War: sem coluna de tags
[ignorado] Western: sem coluna de tags
[ignorado] Mystery: sem coluna de tags
[ignorado] Crime: sem coluna de tags
[ignorado] Comedy: sem coluna de tags
[ignorado] Romance: sem coluna de tags
[ignorado] History: sem coluna de tags
[ignorado] Family: sem coluna de tags

               fonte nomes_steam  idx_score  n_avaliaveis  positivos  prevalencia  ok
genero_imdb                                                                         
Action       genres      action          0        100205      43521        0.434  ok
Adventur

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# 1) Finaliza a avaliação cross-domain nos 3 gêneros com ground-truth direto
aval_sub = avaliavel[usar_jogo]
y_sub = Y_val.loc[usar_jogo].to_numpy()

metricas_val = []
for j, g in enumerate(Y_val.columns):
    mask = aval_sub[:, j]
    idx_c = classes.index(g)
    y_true = y_sub[mask, j]
    y_score = scores[mask, idx_c]
    metricas_val.append({
        "genero": g,
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
    })
print(pd.DataFrame(metricas_val).set_index("genero").to_string(float_format=lambda v: f"{v:.3f}"))

# 2) Probabilidades de gênero (21 dimensões) para TODOS os jogos e filmes
G_jogos = np.zeros((len(jogos), len(classes)), dtype=np.float32)
G_jogos[usar_jogo] = clf.predict_proba(X_desc_jogos[usar_jogo])
G_jogos = normalize(G_jogos)

# Para filmes: combina rótulo real (Y) com probabilidade prevista (suaviza filmes com poucos gêneros)
G_filmes_pred = clf.predict_proba(X_desc)
G_filmes = normalize(0.7 * Y.astype(np.float32) + 0.3 * G_filmes_pred)

print("Matrizes de gênero prontas:", G_jogos.shape, G_filmes.shape)

           roc_auc  pr_auc
genero                    
Action       0.666   0.596
Adventure    0.574   0.486
Sport        0.803   0.382
Matrizes de gênero prontas: (101769, 21) (60875, 21)


In [ ]:
# Pré-calcula filtro de elegibilidade e qualidade normalizada dos filmes
elegivel_filme = (filmes["votes"].fillna(0) >= MIN_VOTOS_FILME).to_numpy()
rating_norm = ((filmes["rating"].fillna(5.0) - 5.0) / 5.0).clip(0, 1).to_numpy()

# Índices rápidos de busca por appID ou nome
appid_to_idx = pd.Series(jogos.index.values, index=jogos["appID"]).to_dict()

def recomendar_filmes(
    biblioteca_steam,
    top_k=10,
    alpha=ALPHA,
    beta=BETA,
    generos_excluir=None,
):
    """
    biblioteca_steam: lista de appIDs [730, 570, ...]
                      OU lista de tuplas [(appID, horas_jogadas), ...]
    """
    if not biblioteca_steam:
        raise ValueError("A biblioteca informada está vazia.")

    # Normaliza entrada para lista de (idx, peso)
    itens = []
    for item in biblioteca_steam:
        if isinstance(item, (tuple, list)):
            appid, horas = item[0], float(item[1])
            peso = np.log1p(max(horas, 0.0)) + 0.5  # log para evitar que 1 jogo de 2000h domine tudo
        else:
            appid, peso = item, 1.0
        if appid in appid_to_idx:
            itens.append((appid_to_idx[appid], peso))

    if not itens:
        raise ValueError("Nenhum appID encontrado na base tratada.")

    idxs = [i for i, _ in itens]
    pesos = np.array([w for _, w in itens], dtype=np.float32).reshape(-1, 1)

    # 1) Centróides do perfil do usuário (texto + gênero)
    u_texto = normalize(np.asarray(X_jogos[idxs].multiply(pesos).sum(axis=0)))
    u_genero = normalize((G_jogos[idxs] * pesos).sum(axis=0, keepdims=True))

    # 2) Similaridades contra o catálogo de filmes
    sim_texto = np.asarray(X_filmes @ u_texto.T).ravel()
    sim_genero = (G_filmes @ u_genero.T).ravel()

    score_final = alpha * sim_texto + (1.0 - alpha) * sim_genero + beta * rating_norm

    # 3) Aplica filtros de elegibilidade
    mascara = elegivel_filme.copy()
    if generos_excluir:
        excluir = set(generos_excluir)
        mascara &= ~filmes["genres"].map(lambda gs: bool(excluir.intersection(gs))).to_numpy()

    score_final = np.where(mascara, score_final, -np.inf)
    top_idx = np.argpartition(-score_final, top_k)[:top_k]
    top_idx = top_idx[np.argsort(-score_final[top_idx])]

    # 4) Monta resultado + explicação (top termos e gêneros que conectaram o perfil ao filme)
    u_texto_arr = np.asarray(u_texto).ravel()
    u_gen_arr = u_genero.ravel()
    resultados = []

    for i in top_idx:
        contrib_termos = u_texto_arr * X_filmes[i].toarray().ravel()
        top_t = [vocab[t] for t in np.argsort(-contrib_termos)[:3] if contrib_termos[t] > 0]

        contrib_gen = u_gen_arr * G_filmes[i]
        top_g = [classes[g] for g in np.argsort(-contrib_gen)[:2] if contrib_gen[g] > 0]

        resultados.append({
            "title": filmes.at[i, "title"],
            "year": filmes.at[i, "release_date"].year if pd.notna(filmes.at[i, "release_date"]) else None,
            "genres": ", ".join(filmes.at[i, "genres"]),
            "rating": filmes.at[i, "rating"],
            "score": round(float(score_final[i]), 4),
            "sim_texto": round(float(sim_texto[i]), 3),
            "sim_genero": round(float(sim_genero[i]), 3),
            "por_que": f"Gêneros: {', '.join(top_g)} | Temas: {', '.join(top_t)}",
        })

    return pd.DataFrame(resultados)

In [ ]:
def buscar_appid(nome_parcial):
    m = jogos[jogos["name"].str.contains(nome_parcial, case=False, na=False)].sort_values(
        "total_reviews", ascending=False
    )
    if m.empty:
        raise ValueError(f"Jogo não encontrado: {nome_parcial}")
    return int(m.iloc[0]["appID"]), m.iloc[0]["name"]

# 1) Sanity check qualitativo (perfil Sci-Fi / Terror Espacial)
lib_teste = [
    buscar_appid("Elden Ring")[0],
    buscar_appid("Hogwarts Legacy")[0],
    buscar_appid("Skyrim")[0],
]
print("Jogos na biblioteca de teste:", [jogos.loc[appid_to_idx[i], "name"] for i in lib_teste])
display(recomendar_filmes(lib_teste, top_k=10))

# 2) Benchmark proxy de franquias / adaptações cruzadas
casos_proxy = [
    ("Alien: Isolation", r"\bAlien\b|Aliens"),
    ("Batman: Arkham City", r"The Dark Knight|Batman"),
    ("Mad Max", r"Mad Max"),
    ("STAR WARS Jedi: Fallen Order", r"Star Wars"),
    ("Shadow of Mordor", r"Lord of the Rings|The Hobbit"),
    ("The Walking Dead", r"28 Days Later|Dawn of the Dead|Night of the Living Dead"),
    ("Deus Ex: Human Revolution|Detroit: Become Human", r"Blade Runner|Ghost in the Shell"),
    ("Red Dead Redemption 2", r"The Good, the Bad and the Ugly|Unforgiven|Django Unchained"),
]

# Pré-resolve os appIDs válidos uma única vez
casos_validos = []
for jogo_query, filme_esperado_rx in casos_proxy:
    try:
        appid, nome_real = buscar_appid(jogo_query)
        casos_validos.append((appid, nome_real, filme_esperado_rx))
    except ValueError as e:
        print(f"[aviso] {e}")

def avaliar_config(alpha_val, beta_val, top_k=20):
    hits, rr = [], []
    for appid, _, filme_esperado_rx in casos_validos:
        recs = recomendar_filmes([appid], top_k=top_k, alpha=alpha_val, beta=beta_val)
        match = recs["title"].str.contains(filme_esperado_rx, case=False, na=False)
        if match.any():
            rank = int(np.argmax(match.to_numpy())) + 1
            hits.append(1)
            rr.append(1.0 / rank)
        else:
            hits.append(0)
            rr.append(0.0)
    return int(np.sum(hits)), float(np.mean(hits)), float(np.mean(rr))

# 3) Varredura de ALPHA (mantendo BETA inicial como referência)
ALPHAS_TESTE = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.25, 0.0]
resultados_alpha = []

for alpha_teste in ALPHAS_TESTE:
    n_hits, hit_rate, mrr = avaliar_config(alpha_teste, BETA, top_k=20)
    resultados_alpha.append({
        "alpha": alpha_teste,
        "hits": f"{n_hits}/{len(casos_validos)}",
        "hit_rate_20": hit_rate,
        "mrr": mrr,
    })

df_eval_alpha = pd.DataFrame(resultados_alpha)

# Critério de seleção: maior HitRate@20, desempate por maior MRR e proximidade ao equilíbrio (0.5)
df_eval_alpha["dist_centro"] = (df_eval_alpha["alpha"] - 0.5).abs()
melhor_alpha_row = df_eval_alpha.sort_values(
    by=["hit_rate_20", "mrr", "dist_centro"],
    ascending=[False, False, True]
).iloc[0]

ALPHA_OTIMO = float(melhor_alpha_row["alpha"])

# 4) Ajuste fino de BETA com o ALPHA_OTIMO selecionado
BETAS_TESTE = [0.0, 0.05, 0.10, 0.15]
resultados_beta = []
for beta_teste in BETAS_TESTE:
    n_hits, hit_rate, mrr = avaliar_config(ALPHA_OTIMO, beta_teste, top_k=20)
    resultados_beta.append({
        "beta": beta_teste,
        "hits": f"{n_hits}/{len(casos_validos)}",
        "hit_rate_20": hit_rate,
        "mrr": mrr,
    })

df_eval_beta = pd.DataFrame(resultados_beta)
melhor_beta_row = df_eval_beta.sort_values(
    by=["hit_rate_20", "mrr"],
    ascending=[False, False]
).iloc[0]

BETA_OTIMO = float(melhor_beta_row["beta"])

# Atualiza as variáveis globais usadas na Etapa 4
ALPHA = ALPHA_OTIMO
BETA = BETA_OTIMO

# 5) Prints explícitos no console com os resultados e parâmetros finais
print("\n" + "=" * 68)
print("AVALIAÇÃO DE SENSIBILIDADE DO PARÂMETRO ALPHA (TOP-20)")
print("=" * 68)
for _, r in df_eval_alpha.iterrows():
    marca = "  <-- SELECIONADO" if r["alpha"] == ALPHA else ""
    print(
        f"ALPHA = {r['alpha']:4.2f} | Acertos: {r['hits']} | "
        f"HitRate@20: {r['hit_rate_20']:6.2%} | MRR: {r['mrr']:.3f}{marca}"
    )

print("\n" + "=" * 68)
print(f"AJUSTE FINO DO PARÂMETRO BETA (COM ALPHA = {ALPHA:.2f})")
print("=" * 68)
for _, r in df_eval_beta.iterrows():
    marca = "  <-- SELECIONADO" if r["beta"] == BETA else ""
    print(
        f"BETA  = {r['beta']:4.2f} | Acertos: {r['hits']} | "
        f"HitRate@20: {r['hit_rate_20']:6.2%} | MRR: {r['mrr']:.3f}{marca}"
    )

print("\n" + "=" * 68)
print("PARÂMETROS FINAIS DO MODELO (CONGELADOS PARA PRODUÇÃO/ETAPA 4)")
print("=" * 68)
parametros_finais = {
    "ALPHA (peso similaridade lexical TF-IDF)": ALPHA,
    "1 - ALPHA (peso similaridade semântica de gêneros)": round(1.0 - ALPHA, 4),
    "BETA (peso bônus de qualidade IMDb)": BETA,
    "PESO_GENEROS (repetição de rótulos no texto)": PESO_GENEROS,
    "MIN_DOC_POR_BASE (frequência mínima cruzada)": MIN_DOC_POR_BASE,
    "MIN_FILMES_POR_GENERO (corte do classificador)": MIN_FILMES_POR_GENERO,
    "MIN_VOTOS_FILME (filtro de popularidade mínima)": MIN_VOTOS_FILME,
    "MIN_TERMOS (densidade mínima do vetor)": MIN_TERMOS,
    "Vocabulário compartilhado (features TF-IDF)": len(vocab),
    "Dimensões latentes de gênero (classes IMDb)": len(classes),
    "Desempenho Proxy Final (HitRate@20)": f"{melhor_beta_row['hit_rate_20']:.2%}",
    "Desempenho Proxy Final (MRR@20)": f"{melhor_beta_row['mrr']:.3f}",
}
for k, v in parametros_finais.items():
    print(f"  - {k:<52}: {v}")
print("=" * 68)

Jogos na biblioteca de teste: ['ELDEN RING', 'Hogwarts Legacy', 'The Elder Scrolls V: Skyrim']


,title,year,genres,rating,score,sim_texto,sim_genero,por_que
0,The Lord of the Rings: The Fellowship of the Ring,2001,"Action Epic, Adventure Epic, Dark Fantasy, Epi...",8.9,0.5722,0.267,0.721,"Gêneros: Fantasy, Adventure | Temas: fantasy e..."
1,"Godzilla, Mothra and King Ghidorah: Giant Mons...",2001,"Action Epic, Dark Fantasy, Dinosaur Adventure,...",7.0,0.5450,0.208,0.802,"Gêneros: Fantasy, Action | Temas: fantasy, epi..."
2,Digimon Adventure tri. Part 1: Reunion,2015,"Anime, Action, Adventure, Animation, Drama, Fa...",7.2,0.5447,0.137,0.865,"Gêneros: Fantasy, Sci-Fi | Temas: fantasy, act..."
3,How to Train Your Dragon: The Hidden World,2019,"Computer Animation, Teen Fantasy, Action, Adve...",7.4,0.5313,0.197,0.770,"Gêneros: Fantasy, Action | Temas: fantasy, fan..."
4,Avatar,2009,"Action Epic, Adventure Epic, Fantasy Epic, Sci...",7.9,0.5310,0.213,0.733,"Gêneros: Fantasy, Sci-Fi | Temas: fantasy, epi..."
5,Ramayana: The Legend of Prince Rama,1997,"Anime, Fantasy Epic, Quest, Sword & Sorcery, A...",9.1,0.5251,0.207,0.679,"Gêneros: Fantasy, Action | Temas: fantasy, fan..."
6,Star Wars: Episode V - The Empire Strikes Back,1980,"Action Epic, Adventure Epic, Dark Fantasy, Epi...",8.7,0.5249,0.178,0.724,"Gêneros: Action, Sci-Fi | Temas: fantasy epic,..."
7,How to Train Your Dragon 2,2014,"Computer Animation, Teen Fantasy, Action, Adve...",7.8,0.5246,0.219,0.718,"Gêneros: Fantasy, Action | Temas: fantasy, fan..."
8,Star Wars: Episode VI - Return of the Jedi,1983,"Action Epic, Adventure Epic, Dark Fantasy, Epi...",8.3,0.5230,0.180,0.734,"Gêneros: Sci-Fi, Action | Temas: fantasy epic,..."
9,Ne Zha 2,2025,"Computer Animation, Fantasy Epic, Martial Arts...",8.2,0.5221,0.214,0.702,"Gêneros: Fantasy, Action | Temas: fantasy, fan..."


[aviso] Jogo não encontrado: Shadow of Mordor

AVALIAÇÃO DE SENSIBILIDADE DO PARÂMETRO ALPHA (TOP-20)
ALPHA = 1.00 | Acertos: 4/7 | HitRate@20: 57.14% | MRR: 0.213
ALPHA = 0.90 | Acertos: 4/7 | HitRate@20: 57.14% | MRR: 0.262
ALPHA = 0.80 | Acertos: 4/7 | HitRate@20: 57.14% | MRR: 0.246
ALPHA = 0.70 | Acertos: 4/7 | HitRate@20: 57.14% | MRR: 0.270
ALPHA = 0.60 | Acertos: 4/7 | HitRate@20: 57.14% | MRR: 0.274
ALPHA = 0.50 | Acertos: 5/7 | HitRate@20: 71.43% | MRR: 0.283  <-- SELECIONADO
ALPHA = 0.25 | Acertos: 4/7 | HitRate@20: 57.14% | MRR: 0.310
ALPHA = 0.00 | Acertos: 3/7 | HitRate@20: 42.86% | MRR: 0.302

AJUSTE FINO DO PARÂMETRO BETA (COM ALPHA = 0.50)
BETA  = 0.00 | Acertos: 3/7 | HitRate@20: 42.86% | MRR: 0.127
BETA  = 0.05 | Acertos: 3/7 | HitRate@20: 42.86% | MRR: 0.211
BETA  = 0.10 | Acertos: 3/7 | HitRate@20: 42.86% | MRR: 0.232
BETA  = 0.15 | Acertos: 5/7 | HitRate@20: 71.43% | MRR: 0.283  <-- SELECIONADO

PARÂMETROS FINAIS DO MODELO (CONGELADOS PARA PRODUÇÃO/ETAPA 4)
  - AL

In [ ]:
import joblib
from scipy import sparse

PASTA_MODELO = "/content/drive/MyDrive/projeto_ia/modelo_final"
os.makedirs(PASTA_MODELO, exist_ok=True)

sparse.save_npz(f"{PASTA_MODELO}/X_jogos.npz", X_jogos)
sparse.save_npz(f"{PASTA_MODELO}/X_filmes.npz", X_filmes)
np.save(f"{PASTA_MODELO}/G_jogos.npy", G_jogos)
np.save(f"{PASTA_MODELO}/G_filmes.npy", G_filmes)

joblib.dump({
    "vectorizer": vectorizer,
    "clf": clf,
    "mlb": mlb,
    "vocab": vocab,
    "config": {"ALPHA": ALPHA, "BETA": BETA, "MIN_VOTOS_FILME": MIN_VOTOS_FILME},
}, f"{PASTA_MODELO}/artefatos_modelo.joblib", compress=3)

# Salva versões enxutas dos DataFrames apenas com as colunas usadas na inferência
jogos[["appID", "name", "genres"]].to_parquet(f"{PASTA_MODELO}/catalogo_jogos.parquet", index=False)
filmes[["id", "title", "genres", "description", "release_date", "rating", "votes"]].to_parquet(
    f"{PASTA_MODELO}/catalogo_filmes.parquet", index=False
)
print("Modelo final salvo com sucesso em:", PASTA_MODELO)

Modelo final salvo com sucesso em: /content/drive/MyDrive/projeto_ia/modelo_final
